# Resources and the method figure (one-off)

Runtime -> Change runtime type -> **T4 GPU** (the machine of most runs). Then **Runtime -> Run all**.

- `scripts/benchmark.py`: time per scan of every preprocessing step (crop + ground removal, NDT cell generation G with and
  without the ground, BEV image) on the CPU, the stored size per scan, and for every network the time per descriptor,
  the peak GPU memory of inference and of one training step with the recipe's batch, the time of a training step and
  from it an epoch / 20-epoch estimate on this GPU (with the descriptor caches for hard negatives), the retrieval time in a
  map of 3000 descriptors, the total time of one localisation from a raw scan, and the exact number of parameters.
  Everything in one table: `comparison.csv`.
  The networks get random weights (time and memory do not depend on the weights).
- `scripts/figure_method.py`: one scan as a point cloud, as 2000 NDT cells and as a BEV image (the report's method figure).

Only the first scans of sequence 08 are downloaded (range requests into the KITTI zip, no Azure). No training, nothing
changes in the runs. Results: `MyDrive/lpr/benchmark/`.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
DRIVE = "/content/drive/MyDrive/lpr"
KITTI = "/content/kitti"
SEQ = "08"
N_SCANS = 50          # scans for the preprocessing timings (about 3-6 s per scan on Colab's CPU)
FIG_IDX = 30          # scan of the method figure (< N_SCANS)
OUT = f"{DRIVE}/benchmark"

## 2. Drive, GPU, code

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
!lscpu | grep -E "Model name|^CPU\(s\)"
url = REPO_URL
try:  # private repo: store a token in Colab Secrets as GITHUB_TOKEN
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
if not os.path.exists("/content/lpr"):
    subprocess.run(["git", "clone", "-q", url, "/content/lpr"], check=True)
else:
    subprocess.run(["git", "-C", "/content/lpr", "pull", "-q"], check=True)
%cd /content/lpr
!git log --oneline -1
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt
os.makedirs(OUT, exist_ok=True)

## 3. The first scans of the sequence

In [ ]:
if len(os.listdir(f"{KITTI}/sequences/{SEQ}/velodyne") if os.path.isdir(f"{KITTI}/sequences/{SEQ}/velodyne") else []) < N_SCANS:
    !python scripts/download_kitti.py --out {KITTI} --seqs {SEQ} --first {N_SCANS} --threads 16
print(len(os.listdir(f"{KITTI}/sequences/{SEQ}/velodyne")), "scans")

## 4. Benchmark

In [ ]:
!python scripts/benchmark.py --kitti_root {KITTI} --seq {SEQ} --n {N_SCANS} --out_dir {OUT}

In [ ]:
import pandas as pd
comp = pd.read_csv(f"{OUT}/comparison.csv")
display(comp.set_index("network").T)

## 5. Method figure

In [ ]:
from IPython.display import Image, display
!python scripts/figure_method.py --kitti_root {KITTI} --seq {SEQ} --idx {FIG_IDX} --out {OUT}/fig_method.png
display(Image(f"{OUT}/fig_method.png", width=1300))